# 记忆治理策略

## 1.消息裁剪

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL
)

In [ ]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:


    messages = state["messages"]

    if len(messages) <= 3:
        return None

    first_message = messages[0]

    # 如果有偶数条消息则取最近的3条消息，如果是奇数条消息，则取最近的4条消息
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]

    new_messages = [first_message] + recent_messages

    return {
        "messages": [
            RemoveMessage(id = REMOVE_ALL_MESSAGES),
            *new_messages   # 解包操作
        ]
    }


agent = create_agent(
    model = model,
    middleware = [trim_messages],
    checkpointer = InMemorySaver(),
)

config : RunnbaleConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": HumanMessage("你好，我是老吴")}, config = config)
agent.invoke({"messages": HumanMessage("从现在起，你叫奶龙")}, config = config)
agent.invoke({"messages": HumanMessage("今天天气不错啊，奶龙")}, config = config)
final_res = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我又是谁？")]},config = config)

for msg in final_res["messages"]:
    msg.pretty_print()

## 2.消息删除

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL
)

In [ ]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    # 保持最近5条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id = m.id) for m in messages[:to_delete]]}
    return None


agent = create_agent(
    model = model,
    middleware = [delete_old_messages],
    checkpointer = InMemorySaver(),
)

config : RunnbaleConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": HumanMessage("你好，我是老吴")}, config = config)
agent.invoke({"messages": HumanMessage("从现在起，你叫奶龙")}, config = config)
agent.invoke({"messages": HumanMessage("今天天气不错啊，奶龙")}, config = config)
final_res = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我又是谁？")]},config = config)

for msg in final_res["messages"]:
    msg.pretty_print()

## 3.消息摘要

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model_out = init_chat_model(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL
)

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model_in = init_chat_model(
    model = "deepseek-v4-pro",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL
)

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# 创建带有摘要中间件的Agent
agent = create_agent(
    model = model_out,
    tools = [],
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model_in,
            trigger = [
                ("tokens", 100), # 超过 100 tokens就摘要
            ],
            keep = ("messages", 2),
            summary_prompt = "对历史消息摘要，消息列表如下\n{messages}",
        )
    ]
)

config = {"configurable": {"thread_id":"1"}}

print("\n进行多轮对话...")
conversations = [
    "我叫江来，一名算法工程师。这里是一段非常长非常长的废话..." * 20, # 强制撑爆 100 tokens
    "请总结一下我的信息"
]

for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    for msg in response["messages"]:
        msg.pretty_print()
    print("*" * 50)

In [ ]:
from rich import print as rprint

final_state = agent.get_state(config)
rprint(final_state)